In [ ]:
%cd ..

import torch

from torch_geometric.datasets import FakeDataset

from nn.net_builder import CustomMlp
from nn.net_builder import CustomGnn

In [4]:
#Create a Custom Mlp Net from parameters

norm_args = {'num_features': 32}
act_args = {}

mlp_args = {'in_size': 10, 'out_size': 3, 'hidden_sz': 16, 'n_blocks': 3, 'norm_type': 'batch',
            'norm_args': norm_args, 'act_type': 'relu', 'act_args': act_args, 'dropout': True,
            'p': 0.2, 'residual': False, 'last_raw': True}

test_mlp = CustomMlp(**mlp_args)

In [5]:
#MLP Layout
test_mlp

CustomMlp(
  (layers): ModuleList(
    (0): LinearBlock(
      <dropout=(True, 0.2), residual=False>
      (lin_layer): Linear(10, 16, bias=True)
      (norm_layer): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act_layer): ReLU()
    )
    (1): LinearBlock(
      <dropout=(True, 0.2), residual=False>
      (lin_layer): Linear(16, 16, bias=True)
      (norm_layer): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act_layer): ReLU()
    )
    (2): LinearBlock(
      <dropout=False, residual=False>
      (lin_layer): Linear(16, 3, bias=True)
    )
  )
)

In [6]:
#Forward Pass Test
input_x = torch.randn((10,10))
test_mlp(input_x)

tensor([[ 0.7927, -1.2001,  1.1153],
        [ 0.4856, -0.1889, -0.0819],
        [ 1.2997, -0.4094,  0.9535],
        [-0.6774, -0.7300,  0.2652],
        [ 0.4087, -0.7441,  1.7818],
        [ 1.0411, -0.3660, -1.0436],
        [ 0.9478, -0.3282, -0.0977],
        [ 0.6948,  0.0765, -0.0593],
        [-0.3138, -1.4761, -0.2147],
        [-1.4370, -0.0159,  0.9089]], grad_fn=<AddmmBackward0>)

In [7]:
#Create a quick fake dataset to get data to test a GNN on
fds = FakeDataset(num_graphs=10, avg_num_nodes=10, avg_degree=2, num_channels=6, edge_dim=3)
fds1 = fds.get(0)
fds1

Data(y=[1], edge_index=[2, 38], x=[12, 6], edge_attr=[38, 3])

In [8]:
#Custom GNN with GCN core

x_feats=6
x_out=3
core_hidden=16
core_blocks=2
encodings='rwpe'
encode_size=4

#Arguments for positional encodings, in this case RWPE
encode_args={'walk_length': encode_size}

#arguments for the Message Passing Blocks
mp_args = {'in_channels': core_hidden, 'out_channels': core_hidden}
gnn_norm_args = {'in_channels': core_hidden}
gnn_act_args = {}
gnn_edge_drop_args = {'p': 0.05, 'force_undirected': True}

mpblock_args = {'mp_type': 'gcn', 'mp_args': mp_args, 'norm_type': 'batch', 'norm_args': gnn_norm_args,
                'act_type': 'relu', 'act_args': gnn_act_args, 'drop_graph': 'edge', 'drop_graph_args': gnn_edge_drop_args,
                'drop_x': True, 'dropx_p': 0.1, 'residual_node': True, 'edge_attr': False}

#Arguments for the Feed Forward Blocks
ff_hidden = 32

ffblock_args = {'in_size': core_hidden, 'out_size': core_hidden, 'hidden_size': ff_hidden, 'norm_type': 'batch',
                'norm_args': {'num_features': core_hidden}, 'act_type': 'relu', 'act_args': {}}

#Combine into Arguments for the 'Core' of the GNN
core_args = {'mp_args': mpblock_args, 'ff_args': ffblock_args}

test_gcn = CustomGnn(x_feats=x_feats, x_out=x_out, core_hidden=core_hidden, core_blocks=core_blocks, core_args=core_args,
                     encodings=encodings, encode_size=encode_size, encode_args=encode_args)


In [9]:
#GNN Layout (truncated)
test_gcn

CustomGnn(
  (preproc): CustomMlp(
    (layers): ModuleList(
      (0): LinearBlock(
        <dropout=False, residual=False>
        (lin_layer): Linear(10, 16, bias=True)
      )
    )
  )
  (gnn_core): ModuleList(
    (0-1): 2 x GnnBlock(
      (message_pass): MessagePassBlock(
        <drop_x=(True, 0.1), residual=True>
        (drop_graph): GraphEdgeDropout(p=0.05, fu=True)
        (mp_layer): GCNConv(16, 16)
        (norm_layer): BatchNorm(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act_layer): ReLU()
      )
      (feed_forward): FeedForwardBlock(
        residual=True
        (lin1): Linear(16, 32, bias=True)
        (act_layer): ReLU()
        (lin2): Linear(32, 16, bias=True)
        (norm_layer): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
  )
  (postproc): CustomMlp(
    (layers): ModuleList(
      (0): LinearBlock(
        <dropout=False, residual=False>
        (lin_layer): Linear(16, 8, bia

In [10]:
#Forward Pass Test
test_gcn(fds1)

tensor([[ 0.5606,  0.2392, -0.3318],
        [-1.1636, -1.7954, -0.2994],
        [-1.9636, -1.9439, -0.2260],
        [-1.8191, -1.8360,  0.2818],
        [-1.5620, -1.8259, -1.2990],
        [-1.1208, -1.2801, -1.0143],
        [ 0.4103,  0.2886, -0.2447],
        [-0.9026, -0.7661, -0.2472],
        [ 0.3423,  0.3844, -0.2704],
        [-1.3943, -1.4252, -0.6233],
        [ 0.5113,  0.3974, -0.2832],
        [ 0.5170,  0.0918, -0.8100]], grad_fn=<AddmmBackward0>)

In [11]:
# GNN Core Details
test_gcn.gnn_core

ModuleList(
  (0-1): 2 x GnnBlock(
    (message_pass): MessagePassBlock(
      <drop_x=(True, 0.1), residual=True>
      (drop_graph): GraphEdgeDropout(p=0.05, fu=True)
      (mp_layer): GCNConv(16, 16)
      (norm_layer): BatchNorm(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act_layer): ReLU()
    )
    (feed_forward): FeedForwardBlock(
      residual=True
      (lin1): Linear(16, 32, bias=True)
      (act_layer): ReLU()
      (lin2): Linear(32, 16, bias=True)
      (norm_layer): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
)

In [12]:
# Postproc Net Details
test_gcn.postproc

CustomMlp(
  (layers): ModuleList(
    (0): LinearBlock(
      <dropout=False, residual=False>
      (lin_layer): Linear(16, 8, bias=True)
      (act_layer): ReLU()
    )
    (1): LinearBlock(
      <dropout=False, residual=False>
      (lin_layer): Linear(8, 4, bias=True)
      (act_layer): ReLU()
    )
    (2): LinearBlock(
      <dropout=False, residual=False>
      (lin_layer): Linear(4, 3, bias=True)
    )
  )
)

In [13]:
# Custom GNN, GAT core, no embeddings, Global Mean Pool
# Multi-head GAT with the 'concat' option does not work with Residual Nets, so the latter is turned off

x_feats=6
x_out=3
core_hidden=16
core_blocks=2
encodings=None
encode_size=0

#Message Passing BLock Args

mp_args = {'in_channels': core_hidden, 'out_channels': core_hidden, 'heads': 3, 'concat': True}
gnn_norm_args = {'in_channels': core_hidden * mp_args['heads']}
gnn_act_args = {}

mpblock_args = {'mp_type': 'gat', 'mp_args': mp_args, 'norm_type': 'graph', 'norm_args': gnn_norm_args,
                'act_type': 'relu', 'act_args': {}, 'drop_x': True, 'dropx_p': 0.1, 'residual_node': False,
                'edge_attr': True}

#Feed Forward Block Args

ff_hidden = 32

ffblock_args = {'in_size': core_hidden * mp_args['heads'], 'out_size': core_hidden, 'hidden_size': ff_hidden, 'norm_type': 'batch',
                'norm_args': {'num_features': core_hidden}, 'act_type': 'relu', 'act_args': {}, 'residual': False}

core_args = {'mp_args': mpblock_args, 'ff_args': ffblock_args}

pool_norm_args = {'in_channels': core_hidden}
pool_args = {'pool_type': 'mean'}

test_gat = CustomGnn(x_feats=x_feats, x_out=x_out, core_hidden=core_hidden, core_blocks=core_blocks, core_args=core_args,
                     core_heads=mp_args['heads'], global_pool=True, gp_args=pool_args)

In [14]:
#GAT Network Layout
test_gat

CustomGnn(
  (preproc): CustomMlp(
    (layers): ModuleList(
      (0): LinearBlock(
        <dropout=False, residual=False>
        (lin_layer): Linear(6, 16, bias=True)
      )
    )
  )
  (gnn_core): ModuleList(
    (0-1): 2 x GnnBlock(
      (message_pass): MessagePassBlock(
        <drop_x=(True, 0.1), residual=False>
        (mp_layer): GATConv(16, 16, heads=3)
        (norm_layer): GraphNorm(48)
        (act_layer): ReLU()
      )
      (feed_forward): FeedForwardBlock(
        residual=False
        (lin1): Linear(48, 32, bias=True)
        (act_layer): ReLU()
        (lin2): Linear(32, 16, bias=True)
        (norm_layer): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
  )
  (gpool): GlobalPoolBlock(type=mean)
  (postproc): CustomMlp(
    (layers): ModuleList(
      (0): LinearBlock(
        <dropout=False, residual=False>
        (lin_layer): Linear(16, 8, bias=True)
        (act_layer): ReLU()
      )
      (1): LinearBlock(
    

In [15]:
#Global Pooling Check
test_gat.gpool

GlobalPoolBlock(type=mean)

In [16]:
#Forward Pass Test
test_gat(fds1)

tensor([[ 0.0220,  0.0780, -0.5182]], grad_fn=<AddmmBackward0>)